# Chapter 9 — Build It Yourself: Precision & Mixed Precision

Same model, half the bits. Here you'll **see** what fp16 and bf16 can and can't represent,
rescue an underflowing gradient with **loss scaling**, flip on **autocast**, and train a model
in mixed precision — watching its loss track fp32 step for step.

> 💡 **Most of this runs fully on a CPU** — number formats behave identically everywhere, so the
> formats, the failures, loss scaling, and autocast all work locally. Only the final training
> **speedup** needs an NVIDIA GPU's Tensor Cores (open in Colab, *Runtime → GPU*). Running a
> ✍️ cell *before* you fill it prints a friendly ⚠️, not an error — that's expected.

In [ ]:
import torch, contextlib, time
import torch.nn as nn

def get_device():
    if torch.cuda.is_available(): return 'cuda'
    if torch.backends.mps.is_available(): return 'mps'
    return 'cpu'

def sync(device):                       # wait for the GPU before timing (Chapter 8's rule)
    if device == 'cuda': torch.cuda.synchronize()
    elif device == 'mps': torch.mps.synchronize()

device = get_device()
print('device:', device, '| (formats work anywhere; the training speedup needs a CUDA GPU)')

## Step 1 — The three formats  ▶️ Run this

A float is **sign · exponent · mantissa**: the exponent sets the *range*, the mantissa sets the
*precision*. fp32 spends 32 bits; fp16 and bf16 each spend 16 — but split them differently.
Run this and read the table.

<details><summary>What am I looking for?</summary>

fp16 and bf16 are **both 2 bytes**, but fp16 has a *bigger mantissa* (finer precision, tiny
range) and bf16 has a *bigger exponent* (fp32's full range, coarse precision). Watch π lose its
digits in both.
</details>

In [ ]:
mant = {'float32': 23, 'float16': 10, 'bfloat16': 7}
print(f"{'format':>9} {'bytes':>6} {'max':>12} {'min normal':>12} {'mantissa':>9}")
for dt in (torch.float32, torch.float16, torch.bfloat16):
    info = torch.finfo(dt)
    name = str(dt).split('.')[-1]
    print(f'{name:>9} {info.bits//8:>6} {info.max:>12.3g} {info.tiny:>12.3g} {mant[name]:>9}')

print('\nstoring pi = 3.14159265 in each (watch the precision shrink):')
for dt in (torch.float32, torch.float16, torch.bfloat16):
    name = str(dt).split('.')[-1]
    print(f'  {name:>9}: {torch.tensor(3.14159265, dtype=dt).item()}')

## Step 2 — Where low precision bites  ▶️ Run this

Two different things go wrong. **Predict each before running:**
1. Store `70000` in fp16 — what happens? (range)
2. Store `1e-8` in fp16 — what happens? (range, the other end)
3. Compute `256 + 0.5` in bf16 — what happens? (precision)

<details><summary>Reveal</summary>

fp16 **overflows** 70000 → `inf` (max is 65504) and **underflows** 1e-8 → `0` (a lost gradient).
bf16 survives both (it kept fp32's range). But bf16 **swamps** `256 + 0.5` → `256.0`: its 7
mantissa bits can't resolve a 0.5 next to a 256, so the small number is simply lost.
</details>

In [ ]:
for label, value in [('70000 (overflow?)', 70000.0), ('1e-8 (underflow?)', 1e-8)]:
    row = {n: torch.tensor(value, dtype=d).item() for n, d in
           [('fp16', torch.float16), ('bf16', torch.bfloat16)]}
    print(f'{label:>20}:  fp16 -> {row["fp16"]!r:>10}   bf16 -> {row["bf16"]!r}')

for name, dt in [('fp16', torch.float16), ('bf16', torch.bfloat16)]:
    s = (torch.tensor(256.0, dtype=dt) + torch.tensor(0.5, dtype=dt)).item()
    note = '  <- swamped! the 0.5 was lost' if s == 256.0 else ''
    print(f'  256 + 0.5 in {name}: {s}{note}')

## Step 3 — Loss scaling: rescue the tiny gradient  ✍️ Your turn

fp16 underflowed `1e-8` to `0`. **Loss scaling** fixes it: multiply *up* into fp16's range
before storing, then divide back out in fp32. Fill the two lines.

<details><summary>Stuck? Show the answer</summary>

```python
scaled_fp16 = torch.tensor(g * scale, dtype=torch.float16)   # scale UP, store in fp16
recovered   = scaled_fp16.float().item() / scale             # unscale in fp32
```
</details>

In [ ]:
g = 1e-8
scale = 1024.0
naive = torch.tensor(g, dtype=torch.float16).item()    # what fp16 does on its own

# ✍️ rescue g: scale it UP into fp16's range and store in fp16, then unscale in fp32
scaled_fp16 = None      # replace
recovered = None        # replace

print(f'naive fp16(1e-8) = {naive}   (lost)')
print(f'recovered        = {recovered}')

In [ ]:
# ▶️ Check your work
try:
    if recovered is None:
        print('⚠️  Fill in the ✍️ cell above (scale up -> fp16 -> unscale), then re-run. (Expected until you do.)')
    elif naive != 0.0:
        print('⚠️  fp16(1e-8) should underflow to 0 here — check `naive`.')
    elif abs(recovered - g) / g < 0.01:
        print(f'✅ Rescued! Scaling recovered {recovered:.3e} ≈ the original 1e-8 (fp16 alone gave {naive}). That is loss scaling.')
    else:
        print(f'⚠️  Not quite — recovered {recovered!r}, expected ≈ {g}. Check the scale then unscale.')
except Exception as e:
    print('❌', e)

## Step 4 — autocast: 16-bit where it's safe  ✍️ Your turn

`torch.autocast` runs ops in the right precision automatically — matmuls in 16-bit, sensitive
ops in fp32. Replace the no-op `nullcontext` with a real autocast block and watch the matmul
come out in **bf16** (even though its inputs are fp32).

<details><summary>Stuck? Show the answer</summary>

```python
ctx = torch.autocast(device_type=device, dtype=torch.bfloat16)
```
</details>

In [ ]:
x = torch.randn(64, 512, device=device)
w = torch.randn(512, 512, device=device)

# ✍️ replace the no-op nullcontext with an autocast(bf16) block
ctx = contextlib.nullcontext()      # replace
with ctx:
    y = x @ w

print(f'inside the block, x @ w came out as: {y.dtype}')

In [ ]:
# ▶️ Check your work
try:
    if y.dtype == torch.bfloat16:
        print('✅ autocast ran the matmul in bf16 — it cast the fp32 inputs for you. (Sensitive ops would stay fp32.)')
    elif y.dtype == torch.float32:
        print('⚠️  Still fp32 — autocast is not on. Replace the nullcontext with '
              'torch.autocast(device_type=device, dtype=torch.bfloat16), then re-run. (Expected until you do.)')
    else:
        print(f'⚠️  Unexpected dtype {y.dtype}.')
except Exception as e:
    print('❌', e)

## Step 5 — Mixed-precision training  ✍️ Your turn

Now the real thing: train the same model in fp32 and in mixed precision (bf16 autocast), from
the **same initialization**, and check the loss curves track. Fill the `ctx` line so the `mixed`
run wraps its forward in autocast.

<details><summary>Stuck? Show the answer</summary>

```python
ctx = torch.autocast(device_type=device, dtype=torch.bfloat16) if mixed else contextlib.nullcontext()
```
</details>

In [ ]:
def run(mixed):
    torch.manual_seed(0)
    m = nn.Sequential(nn.Linear(1024,1024), nn.ReLU(),
                      nn.Linear(1024,1024), nn.ReLU(), nn.Linear(1024,10)).to(device)
    x = torch.randn(512,1024, device=device); y = torch.randint(0,10,(512,), device=device)
    opt = torch.optim.AdamW(m.parameters(), lr=8e-5); lossfn = nn.CrossEntropyLoss()
    scaler = torch.amp.GradScaler(device, enabled=False)   # bf16 needs no scaler
    def one():
        opt.zero_grad()
        # ✍️ when `mixed`, wrap the forward in autocast(bf16); otherwise a no-op nullcontext
        ctx = contextlib.nullcontext()      # replace (see hint)
        with ctx:
            logits = m(x); loss = lossfn(logits, y)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        return logits.dtype, loss.item()
    seen_dtype, _ = one(); sync(device)             # warm up (first step includes setup)
    losses, t0 = [], time.perf_counter()
    for _ in range(30):
        _, l = one(); losses.append(l)
    sync(device)
    return losses, seen_dtype, (time.perf_counter() - t0) / 30 * 1000

fp32_losses, fp32_dt, fp32_ms = run(mixed=False)
mp_losses, mp_dt, mp_ms = run(mixed=True)
print(f'fp32  : logits {fp32_dt}, loss {fp32_losses[0]:.3f} -> {fp32_losses[-1]:.3f}, {fp32_ms:.1f} ms/step')
print(f'mixed : logits {mp_dt}, loss {mp_losses[0]:.3f} -> {mp_losses[-1]:.3f}, {mp_ms:.1f} ms/step')
print(f'speedup here: {fp32_ms / mp_ms:.2f}x  (~1x on CPU/Apple GPU — the real win needs CUDA Tensor Cores)')

In [ ]:
# ▶️ Check your work
try:
    if mp_dt == torch.float32:
        print('⚠️  The mixed run is still fp32 — fill the ✍️ ctx so `mixed` uses '
              'torch.autocast(device_type=device, dtype=torch.bfloat16). (Expected until you do.)')
    elif mp_dt == torch.bfloat16:
        gap = max(abs(a - b) for a, b in zip(fp32_losses, mp_losses))
        assert gap < 0.1, f'curves drifted by {gap}'
        print(f'✅ Mixed precision ran the forward in bf16, and its loss tracked fp32 within {gap:.4f} the whole way — same curve, 16-bit math.')
    else:
        print(f'⚠️  Unexpected dtype {mp_dt}.')
except Exception as e:
    print('❌', e)

## 🎉 You can train in mixed precision now.

You saw what fp16/bf16 can hold, rescued an underflowing gradient with loss scaling, turned on
autocast, and trained a model whose loss curve tracks fp32 — then *timed* both runs (≈1× here;
the real **2× speedup** needs a CUDA GPU's Tensor Cores). The [mini-project](../project/) takes
this to your **GPT** and also measures the **memory** drop.

Next: [Chapter 10 — Distributed](../../10-speed-distributed/), to scale across many GPUs. 🏎️